# Phase 3: cross-dataset generalization and distortion analysis

Run with **Save Version → Save & Run All (Commit)**, GPU on, Internet on. About 2.5–3 hours; the browser can be closed.
When it finishes, download `results_phase3.zip` from the version's **Output** tab.

In [ ]:
%cd /kaggle/working
!rm -rf Learned-Image-Quality-Assessment-Benchmark
!git clone https://github.com/elise-love/Learned-Image-Quality-Assessment-Benchmark.git
%cd Learned-Image-Quality-Assessment-Benchmark
!pip install -q pyiqa timm

# Datasets live in /tmp: /kaggle/working is limited to ~20 GB and is saved as output.
!mkdir -p /tmp/datasets && ln -sfn /tmp/datasets datasets
!python scripts/download_data.py --datasets koniq10k kadid10k livec spaq
!rm -f datasets/*.tgz && rm -rf datasets/SPAQ/TestImage && df -h /tmp | tail -1

# 1. Retrain one seed of each model (same configs as the main experiment).
#    Written to results/phase3_runs so the main results table is untouched.
!python -m src.train --config configs/vit_s16.yaml --seed 0 --set out_dir=results/phase3_runs results_table=results/phase3_runs.csv
!python -m src.train --config configs/resnet50_lr1e-4.yaml --seed 0 --set out_dir=results/phase3_runs results_table=results/phase3_runs.csv

# 2. Zero-shot cross-dataset evaluation (no retraining on target data).
!python -m src.cross_eval --runs vit_s16_seed0 resnet50_lr1e-4_seed0

# 3. Baselines on the same target datasets.
!python scripts/run_baselines.py --config configs/baselines_cross.yaml

# 4. Tables, per-distortion analysis and figures.
!python scripts/analyze_cross.py

# 5. Package results (+ best checkpoints, kept outside the repo results folder).
!mkdir -p /kaggle/working/best_checkpoints && for r in vit_s16_seed0 resnet50_lr1e-4_seed0; do cp checkpoints/$r/best.pth /kaggle/working/best_checkpoints/$r.pth; done
!rm -rf checkpoints datasets
!cd /kaggle/working && zip -qr results_phase3.zip Learned-Image-Quality-Assessment-Benchmark/results
